In [1]:
import math

import numpy as np
import pandas as pd
import plotly.express as px
import random

In [2]:
#inital request's probability of being valid given data, descriptors or input
pval = 1

#component performance
p_c = .75

pi = np.array([[pval], [1-pval]])
C = np.array([[p_c, 0], [1-p_c, 1]])

In [3]:
#observed component performance
C @ pi

array([[0.75],
       [0.25]])

In [4]:
#building 

rng = np.random.default_rng()

n_components = 6


def build_component_matrix(p_c):
    return np.array(np.array([[p_c, 0], [1-p_c, 1]]))


component_ensemble = {}

for i in range(n_components):
    component_id = f"C{i + 1}"
    p_c = rng.uniform(low=0.5, high=0.8)
    component_ensemble[component_id] = build_component_matrix(p_c)


In [5]:
rng = np.random.default_rng(seed=7)

n_users = 100


def build_human_matrix(p_ac, p_ai):
    return np.array([[p_ac, p_ai], [1 - p_ac, 1 - p_ai]])


user_ensemble = {}

for i in range(n_users):
    user_id = f"u{i + 1}"
    p_ac = 1
    p_ai = 0
    user_ensemble[user_id] = build_human_matrix(p_ac, p_ai)

In [6]:
def link_system_components(ensemble, component_list):
    feature_dict = {}
    result = np.eye(2)
    for key in component_list:
        isSuccess = rng.random() < ensemble[key][0][0]
        feature_dict[key] = isSuccess
        result = np.matmul(result, build_component_matrix(isSuccess))
        
    return result, feature_dict

In [7]:
def simulate_daily_activity(day, n_items, user_ensemble, component_ensemble):
    pi = np.array([[1],[0]])
    telemetry = []
    user_ids = list(user_ensemble.keys())
    component_ids = list(component_ensemble.keys())

    for _ in range(n_items):
        user = rng.choice(user_ids)
        system_inference_path = random.sample(component_ids, random.randint(2,6))
        composite_system_performance, feature_dict = link_system_components(component_ensemble, system_inference_path)
        accept_prob = float((user_ensemble[user] @ composite_system_performance @ pi)[0, 0])
        is_accepted = rng.random() < accept_prob
        telemetry.append(
            [day, user, is_accepted, system_inference_path, feature_dict]
        )

    return pd.DataFrame(
        telemetry,
        columns=[
            "Date",
            "User",
            "isAccepted",
            "systemInferencePath",
            "featuresDict",
        ],
    )

In [8]:
#improvement rate measures as (1-c11)*r per execusion (cadence controlled in the simulation step)
def component_improvement_sprint(component_ensemble, component_list, improvement_rate):
    for key in component_list:
        improvement = (1-component_ensemble[key][0][0])*improvement_rate
        component_ensemble[key][0][0] += improvement
        component_ensemble[key][1][0] -= improvement
    return component_ensemble

In [9]:
simulate_daily_activity(1,1000,user_ensemble, component_ensemble)

,Date,User,isAccepted,systemInferencePath,featuresDict
0,1,u95,False,"[C5, C6, C1, C2]","{'C5': False, 'C6': False, 'C1': True, 'C2': T..."
1,1,u63,False,"[C4, C6, C3, C1, C2]","{'C4': True, 'C6': False, 'C3': False, 'C1': T..."
2,1,u72,False,"[C1, C6, C2, C5, C3]","{'C1': True, 'C6': True, 'C2': True, 'C5': Fal..."
3,1,u26,False,"[C5, C1, C2, C6, C3]","{'C5': False, 'C1': True, 'C2': True, 'C6': Fa..."
4,1,u15,False,"[C3, C1, C2, C6, C4]","{'C3': True, 'C1': False, 'C2': True, 'C6': Tr..."
...,...,...,...,...,...
995,1,u25,False,"[C2, C5, C1, C3, C6]","{'C2': False, 'C5': True, 'C1': False, 'C3': T..."
996,1,u94,False,"[C4, C3, C1, C6]","{'C4': True, 'C3': True, 'C1': True, 'C6': False}"
997,1,u95,True,"[C2, C5, C1]","{'C2': True, 'C5': True, 'C1': True}"
998,1,u40,False,"[C6, C5, C4, C3, C1]","{'C6': True, 'C5': False, 'C4': False, 'C3': F..."


In [10]:
n_items_per_day = 1000
simulation_duration = 365 + 90

historical_telemetry = []

for day_idx in range(simulation_duration):
    # system_correct_prob = min(system_quality_over_time(day_idx), 1.0)
    historical_telemetry.append(
        simulate_daily_activity(
            day=day_idx + 1,
            n_items=n_items_per_day,
            user_ensemble=user_ensemble,
            component_ensemble=component_ensemble,
        )
    )

    if (day_idx + 1) % 7 == 0:
        if day_idx < 365 + 60:
            component_ensemble = component_improvement_sprint(component_ensemble, random.sample(list(component_ensemble.keys()), 1), .4)

historical_telemetry_df = pd.concat(historical_telemetry, ignore_index=True)
historical_telemetry_df.head()

,Date,User,isAccepted,systemInferencePath,featuresDict
0,1,u7,False,"[C1, C2, C4, C3, C6, C5]","{'C1': True, 'C2': True, 'C4': True, 'C3': Tru..."
1,1,u5,False,"[C2, C6, C5, C3, C4, C1]","{'C2': True, 'C6': True, 'C5': True, 'C3': Tru..."
2,1,u26,False,"[C5, C6, C1, C4]","{'C5': False, 'C6': True, 'C1': False, 'C4': T..."
3,1,u57,False,"[C6, C5, C4]","{'C6': False, 'C5': True, 'C4': True}"
4,1,u21,False,"[C1, C2, C6, C3, C4]","{'C1': True, 'C2': True, 'C6': False, 'C3': Tr..."


In [25]:
historical_telemetry_df.systemInferencePath.apply(lambda x: x.sort())

0         None
1         None
2         None
3         None
4         None
          ... 
454995    None
454996    None
454997    None
454998    None
454999    None
Name: systemInferencePath, Length: 455000, dtype: object

In [26]:
observed_daily_accepts = (
    historical_telemetry_df.groupby(["Date"])["isAccepted"]
    .mean()
    .reset_index()
)

fig = px.scatter(
    observed_daily_accepts,
    x="Date",
    y="isAccepted"
)
fig.update_traces(marker={"size": 5})
fig.update_layout(yaxis_tickformat=".0%")
fig.show()

In [19]:
# expanded_df = historical_telemetry_df.systemInferencePath.apply(pd.Series)

expanded_df = pd.json_normalize(historical_telemetry_df.featuresDict)

# Optional: Rename the new columns (e.g., C_1, C_2...)
# expanded_df = expanded_df.rename(columns=lambda x: f'Component_{x+1}')

In [28]:
# expanded_df = ~expanded_df.isna()

In [20]:
list(expanded_df.columns)

['C1', 'C2', 'C4', 'C3', 'C6', 'C5']

In [21]:
historical_telemetry_df[list(expanded_df.columns)] = expanded_df

In [29]:
categories = {}
cc = 1
for inf_path in historical_telemetry_df.systemInferencePath.astype('str').unique():
    categories[inf_path] = "Type_" + str(cc)
    cc += 1

In [30]:
historical_telemetry_df["request_type"] = historical_telemetry_df.systemInferencePath.apply(lambda x: categories[str(x)])

In [31]:
historical_telemetry_df

,Date,User,isAccepted,systemInferencePath,featuresDict,C1,C2,C4,C3,C6,C5,request_type
0,1,u7,False,"[C1, C2, C3, C4, C5, C6]","{'C1': True, 'C2': True, 'C4': True, 'C3': Tru...",True,True,True,True,True,False,Type_1
1,1,u5,False,"[C1, C2, C3, C4, C5, C6]","{'C2': True, 'C6': True, 'C5': True, 'C3': Tru...",False,True,True,True,True,True,Type_1
2,1,u26,False,"[C1, C4, C5, C6]","{'C5': False, 'C6': True, 'C1': False, 'C4': T...",False,NaN,True,NaN,True,False,Type_2
3,1,u57,False,"[C4, C5, C6]","{'C6': False, 'C5': True, 'C4': True}",NaN,NaN,True,NaN,False,True,Type_3
4,1,u21,False,"[C1, C2, C3, C4, C6]","{'C1': True, 'C2': True, 'C6': False, 'C3': Tr...",True,True,True,True,False,NaN,Type_4
...,...,...,...,...,...,...,...,...,...,...,...,...
454995,455,u33,True,"[C2, C3, C4, C5, C6]","{'C2': True, 'C5': True, 'C3': True, 'C6': Tru...",NaN,True,True,True,True,True,Type_20
454996,455,u84,True,"[C2, C5, C6]","{'C6': True, 'C5': True, 'C2': True}",NaN,True,NaN,NaN,True,True,Type_25
454997,455,u87,True,"[C1, C2, C4]","{'C1': True, 'C4': True, 'C2': True}",True,True,True,NaN,NaN,NaN,Type_6
454998,455,u63,True,"[C1, C4]","{'C4': True, 'C1': True}",True,NaN,True,NaN,NaN,NaN,Type_44


In [56]:
daily_accept_stats = (
    historical_telemetry_df.groupby(["Date", "request_type"], as_index=False)
    .agg(
        accepted_sum=("isAccepted", "sum"),
        n_obs=("isAccepted", "size"),
    )
)

if pd.api.types.is_datetime64_any_dtype(daily_accept_stats["Date"]):
    all_dates = pd.date_range(
        daily_accept_stats["Date"].min(),
        daily_accept_stats["Date"].max(),
        freq="D",
        name="Date",
    )
else:
    all_dates = pd.Index(
        range(daily_accept_stats["Date"].min(), daily_accept_stats["Date"].max() + 1),
        name="Date",
    )

observed_daily_accepts = (
    daily_accept_stats.set_index(["request_type", "Date"])
    .reindex(
        pd.MultiIndex.from_product(
            [daily_accept_stats["request_type"].unique(), all_dates],
            names=["request_type", "Date"],
        ),
        fill_value=0,
    )
    .reset_index()
    .sort_values(["request_type", "Date"])
)

observed_daily_accepts[["accepted_sum_7d", "n_obs_7d"]] = (
    observed_daily_accepts.groupby("request_type")[["accepted_sum", "n_obs"]]
    .transform(lambda s: s.rolling(window=7, min_periods=1).sum())
)

observed_daily_accepts["isAccepted"] = (
    observed_daily_accepts["accepted_sum_7d"] / observed_daily_accepts["n_obs_7d"]
)

observed_daily_accepts = observed_daily_accepts[observed_daily_accepts["n_obs"] > 0]

fig = px.scatter(
    observed_daily_accepts,
    x="Date",
    y="isAccepted",
    color="request_type",
    hover_name="request_type",
    opacity=0.45,
    title="7-Day Trailing Acceptance Rate by Request Type",
    labels={"isAccepted": "Acceptance Rate", "Date": "Day"},
)
fig.update_traces(marker={"size": 5})
# fig.update_layout(yaxis_tickformat=".0%")
fig.show()

In [38]:
select_telemetry = historical_telemetry_df.loc[(historical_telemetry_df.Date <= historical_telemetry_df.Date.max()) & (historical_telemetry_df.Date >= historical_telemetry_df.Date.max() - 30)]

In [55]:
select_telemetry

,Date,User,isAccepted,systemInferencePath,featuresDict,C1,C2,C4,C3,C6,C5,request_type
424000,425,u95,True,"[C3, C4]","{'C3': True, 'C4': True}",NaN,NaN,True,True,NaN,NaN,Type_11
424001,425,u8,True,"[C3, C6]","{'C3': True, 'C6': True}",NaN,NaN,NaN,True,True,NaN,Type_46
424002,425,u50,True,"[C2, C4, C5, C6]","{'C5': True, 'C2': True, 'C4': True, 'C6': True}",NaN,True,True,NaN,True,True,Type_7
424003,425,u14,True,"[C1, C2, C3, C4, C5]","{'C1': True, 'C3': True, 'C4': True, 'C5': Tru...",True,True,True,True,NaN,True,Type_27
424004,425,u96,True,"[C1, C2, C3, C6]","{'C3': True, 'C1': True, 'C2': True, 'C6': True}",True,True,NaN,True,True,NaN,Type_28
...,...,...,...,...,...,...,...,...,...,...,...,...
454995,455,u33,True,"[C2, C3, C4, C5, C6]","{'C2': True, 'C5': True, 'C3': True, 'C6': Tru...",NaN,True,True,True,True,True,Type_20
454996,455,u84,True,"[C2, C5, C6]","{'C6': True, 'C5': True, 'C2': True}",NaN,True,NaN,NaN,True,True,Type_25
454997,455,u87,True,"[C1, C2, C4]","{'C1': True, 'C4': True, 'C2': True}",True,True,True,NaN,NaN,NaN,Type_6
454998,455,u63,True,"[C1, C4]","{'C4': True, 'C1': True}",True,NaN,True,NaN,NaN,NaN,Type_44


In [40]:
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
import numpy as np
import pandas as pd

df = select_telemetry.copy()
y = df["isAccepted"].astype(int)
X = df[['User']]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y
)

cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

model = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",  # good for your 7.5% positive rate
    depth=6,
    learning_rate=0.05,
    iterations=2000,
    random_seed=42,
    verbose=200
)

model.fit(
    X_train, y_train,
    cat_features=cat_cols,          # <--- names, not indices
    eval_set=(X_test, y_test),
    use_best_model=True
)

0:	test: 0.5000000	best: 0.5000000 (0)	total: 60.5ms	remaining: 2m
200:	test: 0.5016298	best: 0.5016298 (167)	total: 1.43s	remaining: 12.8s
400:	test: 0.4873900	best: 0.5016298 (167)	total: 3.02s	remaining: 12s
600:	test: 0.4873900	best: 0.5016298 (167)	total: 4.61s	remaining: 10.7s
800:	test: 0.4873900	best: 0.5016298 (167)	total: 6s	remaining: 8.99s
1000:	test: 0.4873900	best: 0.5016298 (167)	total: 7.47s	remaining: 7.46s
1200:	test: 0.4873900	best: 0.5016298 (167)	total: 8.88s	remaining: 5.9s
1400:	test: 0.4873900	best: 0.5016298 (167)	total: 10.2s	remaining: 4.35s
1600:	test: 0.4873900	best: 0.5016298 (167)	total: 11.5s	remaining: 2.86s
1800:	test: 0.4873900	best: 0.5016298 (167)	total: 12.8s	remaining: 1.41s
1999:	test: 0.4873900	best: 0.5016298 (167)	total: 14.1s	remaining: 0us

bestTest = 0.5016298371
bestIteration = 167

Shrink model to first 168 iterations.


In [41]:
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
import numpy as np
import pandas as pd

df = select_telemetry.copy()
y = df["isAccepted"].astype(int)
X = df[['Date','User']]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y
)

cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

model = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",  # good for your 7.5% positive rate
    depth=6,
    learning_rate=0.05,
    iterations=1000,
    # random_seed=42,
    verbose=200
)

model.fit(
    X_train, y_train,
    cat_features=cat_cols,          # <--- names, not indices
    eval_set=(X_test, y_test),
    use_best_model=True
)

0:	test: 0.4826470	best: 0.4826470 (0)	total: 8.1ms	remaining: 8.1s
200:	test: 0.5271239	best: 0.5562167 (11)	total: 1.63s	remaining: 6.47s
400:	test: 0.5378770	best: 0.5562167 (11)	total: 3.44s	remaining: 5.14s
600:	test: 0.5285963	best: 0.5562167 (11)	total: 5.22s	remaining: 3.46s
800:	test: 0.5168763	best: 0.5562167 (11)	total: 7s	remaining: 1.74s
999:	test: 0.5074989	best: 0.5562167 (11)	total: 8.75s	remaining: 0us

bestTest = 0.5562167159
bestIteration = 11

Shrink model to first 12 iterations.


In [42]:
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
import numpy as np
import pandas as pd

df = select_telemetry.copy()
y = df["isAccepted"].astype(int)
X = df[['Date','User', 'request_type']]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y
)

cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()


model = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
    depth=6,
    learning_rate=0.05,
    iterations=1000,
    # random_seed=42,
    verbose=200
)

model.fit(
    X_train, y_train,
    cat_features=cat_cols,
    eval_set=(X_test, y_test),
    use_best_model=True
)

0:	test: 0.5728090	best: 0.5728090 (0)	total: 6.54ms	remaining: 6.53s
200:	test: 0.5559056	best: 0.5938576 (2)	total: 1.66s	remaining: 6.6s
400:	test: 0.5293950	best: 0.5938576 (2)	total: 3.68s	remaining: 5.5s
600:	test: 0.5033619	best: 0.5938576 (2)	total: 5.91s	remaining: 3.92s
800:	test: 0.4931433	best: 0.5938576 (2)	total: 8.21s	remaining: 2.04s
999:	test: 0.4899144	best: 0.5938576 (2)	total: 10.6s	remaining: 0us

bestTest = 0.5938576307
bestIteration = 2

Shrink model to first 3 iterations.


In [54]:
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
import numpy as np
import pandas as pd

df = select_telemetry.copy()
# df[['Component_1','Component_2', 'Component_3', 'Component_4', 'Component_5','Component_6']] = df[['Component_1','Component_2', 'Component_3', 'Component_4', 'Component_5','Component_6']].astype(int)
y = df["isAccepted"].astype(int)
X = df[['Date', 'User', 'C1','C2', 'C3', 'C4', 'C5','C6', 'request_type']]
X.fillna(-1, inplace=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y
)

cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

model = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
    depth=6,
    learning_rate=0.05,
    iterations=2000,
    random_seed=42,
    verbose=200
)

model.fit(
    X_train, y_train,
    cat_features=cat_cols,
    eval_set=(X_test, y_test),
    use_best_model=True
)

/tmp/ipykernel_25689/458770668.py:10: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



0:	test: 1.0000000	best: 1.0000000 (0)	total: 20.2ms	remaining: 40.4s
200:	test: 1.0000000	best: 1.0000000 (0)	total: 3s	remaining: 26.9s
400:	test: 1.0000000	best: 1.0000000 (0)	total: 5.61s	remaining: 22.4s
600:	test: 1.0000000	best: 1.0000000 (0)	total: 8.03s	remaining: 18.7s
800:	test: 1.0000000	best: 1.0000000 (0)	total: 10.4s	remaining: 15.6s
1000:	test: 1.0000000	best: 1.0000000 (0)	total: 12.7s	remaining: 12.7s
1200:	test: 1.0000000	best: 1.0000000 (0)	total: 15.1s	remaining: 10.1s
1400:	test: 1.0000000	best: 1.0000000 (0)	total: 17.5s	remaining: 7.48s
1600:	test: 1.0000000	best: 1.0000000 (0)	total: 19.9s	remaining: 4.96s
1800:	test: 1.0000000	best: 1.0000000 (0)	total: 22.2s	remaining: 2.45s
1999:	test: 1.0000000	best: 1.0000000 (0)	total: 24.6s	remaining: 0us

bestTest = 1
bestIteration = 0

Shrink model to first 1 iterations.


In [ ]:
#Seems like model path is not enough for errors to be separable even with perfect humans